In [4]:
# imports
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, 
    mean_squared_error, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report, 
    roc_curve, 
    roc_auc_score
)
from sklearn.utils import resample
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [12]:
# load all cleaned dfs
clean_2018 = pd.read_csv('clean_2018.csv')
clean_2019 = pd.read_csv('clean_2019.csv')
clean_2020 = pd.read_csv('clean_2020.csv')
clean_2021 = pd.read_csv('clean_2021.csv')
clean_2022 = pd.read_csv('clean_2022.csv')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86659/1019276897.py:3: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  clean_2019 = pd.read_csv('clean_2019.csv')
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86659/1019276897.py:6: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  clean_2022 = pd.read_csv('clean_2022.csv')


In [13]:
# function to rename variables in preparation for making it a dummy variable
def rename_variable(df):

    # add 000 to make it the baseline
    # change to make numbers into categorical strings
    # make dictionary 
    columns_to_rename = {
        'action_taken': {1: 1, 3: 0},
        'purchaser_type': {0: 'Not applicable', 1: 'Fannie Mae', 2: 'Ginnie Mae', 3: 'Freddie Mac', 4: 'Farmer Mac', 5: 'Private securitizer', 6: 'Commercial bank, savings bank, or savings association', 71: 'Credit union, mortgage company, or finance company', 72: 'Life insurance company', 8: 'Affiliate institution', 9: 'Other type of purchaser'},
        'preapproval': {1: 'Preapproval requested', 2: 'Preapproval not requested'},
        'loan_type': {1: 'Conventional (not insured or guaranteed by FHA, VA, RHS, or FSA)', 2: 'Federal Housing Administration insured (FHA)', 3: 'Veterans Affairs guaranteed (VA)', 4: 'USDA Rural Housing Service or Farm Service Agency guaranteed (RHS or FSA)'},
        'lien_status': {1: 'Secured by a first lien', 2: 'Secured by a subordinate lien'},
        'reverse_mortgage': {1: 'Reverse mortgage', 2: 'Not a reverse mortgage', 1111: 'Exempt'},
        'open-end_line_of_credit': {1: 'Open-end line of credit', 2: 'Not an open-end line of credit', 1111: 'Exempt'},
        'hoepa_status': {1: 'High-cost mortgage', 2: 'Not a high-cost mortgage', 3: 'Not applicable'},
        'negative_amortization': {1: 'Negative amortization', 2: 'No negative amortization', 1111: 'Exempt'},
        'interest_only_payment': {1: 'Interest-only payments', 2: 'No interest-only payments', 1111: 'Exempt'},
        'balloon_payment': {1: 'Balloon payment', 2: 'No balloon payment', 1111: 'Exempt'},
        'other_nonamortizing_features': {1: 'Other non-fully amortizing features', 2: 'No other non-fully amortizing features', 1111: 'Exempt'},
        'construction_method': {1: 'Site-built', 2: 'Manufactured home'},
        'manufactured_home_secured_property_type': {1: 'Manufactured home and land', 2: 'Manufactured home and not land', 3: 'Not applicable', 1111: 'Exempt'},
        'manufactured_home_land_property_interest': {1: 'Direct ownership', 2: 'Indirect ownership', 3: 'Paid leasehold', 4: 'Unpaid leasehold', 5: 'Not applicable', 1111: 'Exempt'},
        'total_units': {1: '1', 2: '2', 3: '3', 4: '4'},
        'applicant_credit_score_type': {1: 'Equifax Beacon 5.0', 2: 'Experian Fair Isaac', 3: 'FICO Risk Score Classic 04', 4: 'FICO Risk Score Classic 98', 5: 'VantageScore 2.0', 6: 'VantageScore 3.0', 7: 'More than one credit scoring model', 8: 'Other credit scoring model', 9: 'Not applicable', 1111: 'Exempt'},
        'co-applicant_credit_score_type': {1: 'Equifax Beacon 5.0', 2: 'Experian Fair Isaac', 3: 'FICO Risk Score Classic 04', 4: 'FICO Risk Score Classic 98', 5: 'VantageScore 2.0', 6: 'VantageScore 3.0', 7: 'More than one credit scoring model', 8: 'Other credit scoring model', 9: 'Not applicable', 10: 'No co-applicant', 1111: 'Exempt'},
        'applicant_ethnicity-1': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable'},
        'applicant_ethnicity-2': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity-3': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity-4': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity-5': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-1': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable', 5: 'No co-applicant'},
        'co-applicant_ethnicity-2': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-3': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-4': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'co-applicant_ethnicity-5': {1: 'Hispanic or Latino', 2: '000 Not Hispanic or Latino'},
        'applicant_ethnicity_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable'},
        'co-applicant_ethnicity_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable', 4: 'No co-applicant'},
        'applicant_race-1': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White', 6: 'Information not provided by applicant in mail, internet, or telephone application', 7: 'Not applicable'},
        'applicant_race-2': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race-3': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race-4': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race-5': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-1': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White', 6: 'Information not provided by applicant in mail, internet, or telephone application', 7: 'Not applicable', 8: 'No co-applicant'},
        'co-applicant_race-2': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-3': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-4': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'co-applicant_race-5': {1: 'American Indian or Alaska Native', 2: 'Asian', 3: 'Black or African American', 4: 'Native Hawaiian or Other Pacific Islander', 5: '000 White'},
        'applicant_race_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable'},
        'co-applicant_race_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable', 4: 'No co-applicant'},
        'applicant_sex': {1: '000 Male', 2: 'Female', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable', 6: 'Applicant selected both male and female'},
        'co-applicant_sex': {1: '000 Male', 2: 'Female', 3: 'Information not provided by applicant in mail, internet, or telephone application', 4: 'Not applicable', 5: 'No co-applicant', 6: 'Co-applicant selected both male and female'},
        'applicant_sex_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable'},
        'co-applicant_sex_observed': {1: 'Collected on the basis of visual observation or surname', 2: 'Not collected on the basis of visual observation or surname', 3: 'Not applicable', 4: 'No co-applicant'},
        'submission_of_application': {1: 'Submitted directly to your institution', 2: 'Not submitted directly to your institution', 3: 'Not applicable', 1111: 'Exempt'},
        'initially_payable_to_institution': {1: 'Initially payable to your institution', 2: 'Not initially payable to your institution', 3: 'Not applicable', 1111: 'Exempt'},
        'aus-1': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System', 1111: 'Exempt'},
        'aus-2': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'aus-3': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'aus-4': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'aus-5': {1: 'Desktop Underwriter (DU)', 2: 'Loan Prospector (LP) or Loan Product Advisor', 3: 'Technology Open to Approved Lenders (TOTAL) Scorecard', 4: 'Guaranteed Underwriting System (GUS)', 5: 'Other', 6: 'Not applicable', 7: 'Internal Proprietary System'},
        'derived_ethnicity': {'Not Hispanic or Latino': '000 Not Hispanic or Latino'},
        'derived_race': {'White': '000 White'},
        'derived_sex': {'Male': '000 Male'}}
    
    # apply dictionary
    for column, replacements in columns_to_rename.items():
        if column in df.columns:
            df[column] = df[column].replace(replacements)
    return df

In [14]:
# function that finds columns to based on alphabetic order and prefix 
def drop_first_alphabetically(bool_columns):
    
    # dictionary to group columns by prefix
    prefix_groups = {}

    # group columns by prefix
    for col in bool_columns:
        prefix = col.rsplit('_', 1)[0]
        if prefix not in prefix_groups:
            prefix_groups[prefix] = []
        prefix_groups[prefix].append(col)

    # list of columns to drop
    columns_to_drop = []

    # find which columns to drop based on alphabetic order
    for prefix, columns in prefix_groups.items():
        if len(columns) > 1:
            columns.sort() 
            columns_to_drop.append(columns[0]) 
    return columns_to_drop

In [15]:
# function to create dummy variables
def dummy(df):
    
    # make categorical columns into dummies
    dummy_df = pd.get_dummies(df, drop_first=False)
    
    # drop columns to prevent collinearity 
    bool_columns = dummy_df.select_dtypes(include=bool).columns
    drop_columns = drop_first_alphabetically(bool_columns)
    keep_columns = list(set(bool_columns) - set(drop_columns))
    dummy_df = dummy_df.drop(columns=drop_columns) 
    dummy_df[keep_columns] = dummy_df[keep_columns].astype(int)
    return dummy_df

In [20]:
# function that combines previous functions
def save_dummy_df(df, year):
    dummy_df = dummy(rename_variable(df.dropna()))
    dummy_df.to_csv(f'dummy_{year}.csv', index=False)

In [21]:
# apply combined function on cleaned df
save_dummy_df(clean_2018, '2018')
save_dummy_df(clean_2019, '2019')
save_dummy_df(clean_2020, '2020')
save_dummy_df(clean_2021, '2021')
save_dummy_df(clean_2022, '2022')

/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86659/3602292164.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].replace(replacements)
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86659/3602292164.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].replace(replacements)
/var/folders/b9/kvwg4mzj5kv0208_l9z2ljsc0000gn/T/ipykernel_86659/3602292164.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

## Examine whether the dropped values exhibit random or non-random patterns

In [28]:
#2018
clean_2018.describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,81089.000000,83952.000000,...,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000,83952.000000
mean,1.208071,13.330498,1.936440,1.434046,1.004336,1.999929,2.087776,198322.970269,90.779423,2.117579,...,1.203950,1.172253,1.930579,5040.605858,36.437169,81544.237183,106.294359,1318.726808,1600.970852,45.893344
std,0.610617,24.876228,0.243969,0.670793,0.065704,0.008454,10.126544,89303.900299,9.179245,0.323697,...,6.644693,6.647907,10.212439,1667.431250,27.302910,3213.090942,33.649599,550.434046,623.441601,18.189685
min,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.260000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,67900.000000,14.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,135000.000000,81.840000,2.000000,...,1.000000,1.000000,1.000000,3851.000000,14.870000,82400.000000,82.000000,932.000000,1182.000000,33.000000
50%,1.000000,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,185000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4898.000000,28.510000,82400.000000,105.000000,1291.000000,1594.000000,44.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,255000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6153.000000,53.120000,82400.000000,128.000000,1664.000000,2004.000000,59.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.000000,1111.000000,495000.000000,120.950000,3.000000,...,1111.000000,1111.000000,1111.000000,10370.000000,99.720000,89000.000000,201.000000,2816.000000,3274.000000,76.000000


In [29]:
clean_2018.dropna().describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000,79639.0,79639.000000,79639.000000,79639.000000,79639.000000,...,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000,79639.000000
mean,1.197642,12.963410,1.940657,1.414809,1.004294,2.0,1.995404,198559.688093,90.815380,2.112257,...,1.212622,1.173583,1.826065,5036.644132,36.667827,81610.006404,106.262020,1315.202200,1594.762980,45.980562
std,0.596847,24.484875,0.236267,0.654435,0.065391,0.0,0.067636,89330.123511,9.170979,0.317152,...,6.821977,6.824612,1.275823,1667.581102,27.334869,3172.216866,33.740693,550.781551,622.152935,18.218605
min,1.000000,0.000000,1.000000,1.000000,1.000000,2.0,1.000000,5000.000000,55.260000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,67900.000000,14.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.0,2.000000,135000.000000,82.857000,2.000000,...,1.000000,1.000000,1.000000,3848.000000,14.990000,82400.000000,82.000000,928.000000,1177.000000,33.000000
50%,1.000000,3.000000,2.000000,1.000000,1.000000,2.0,2.000000,185000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4889.000000,28.660000,82400.000000,104.000000,1289.000000,1587.000000,44.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.0,2.000000,255000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6153.000000,53.490000,82400.000000,128.000000,1662.000000,1999.000000,59.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.0,2.000000,495000.000000,120.950000,3.000000,...,1111.000000,1111.000000,6.000000,10370.000000,99.720000,89000.000000,201.000000,2816.000000,3274.000000,76.000000


In [30]:
#2019
clean_2019.describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,62937.000000,65646.000000,...,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000,65646.000000
mean,1.196722,13.128218,1.957027,1.407717,1.006703,1.999939,2.130107,216408.920574,90.795119,2.121774,...,1.178472,1.091917,1.888203,4983.829936,40.284275,81923.102702,107.965649,1283.731149,1527.316303,46.584087
std,0.595609,24.572189,0.202798,0.627747,0.081595,0.007806,12.242172,90040.874676,9.090549,0.328978,...,0.382913,0.347463,1.337168,1619.801312,27.746971,235.808165,35.751416,541.315856,613.218795,18.464557
min,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.250000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,81200.000000,15.000000,17.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,155000.000000,84.990000,2.000000,...,1.000000,1.000000,1.000000,3815.000000,17.670000,82000.000000,82.000000,915.000000,1118.000000,33.000000
50%,1.000000,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,205000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4879.000000,33.060000,82000.000000,106.000000,1265.000000,1521.000000,46.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,275000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6100.000000,57.070000,82000.000000,132.000000,1637.000000,1927.000000,60.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.000000,1111.000000,525000.000000,121.000000,3.000000,...,2.000000,3.000000,6.000000,9980.000000,99.670000,82000.000000,210.000000,2760.000000,3079.000000,76.000000


In [31]:
clean_2019.dropna().describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000,62026.0,62026.000000,62026.000000,62026.000000,62026.000000,...,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000,62026.000000
mean,1.189533,12.795038,1.955503,1.382066,1.006642,2.0,2.138168,216704.446522,90.796009,2.119305,...,1.185326,1.091623,1.806549,4978.539306,40.404706,81928.700867,108.070535,1280.345516,1520.073663,46.720988
std,0.585790,24.173651,0.206199,0.604215,0.081230,0.0,12.594290,90061.272162,9.090736,0.325589,...,0.388565,0.345555,1.200710,1618.748245,27.720625,227.939855,35.870021,541.086166,612.250303,18.467610
min,1.000000,0.000000,1.000000,1.000000,1.000000,2.0,1.000000,5000.000000,55.250000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,81200.000000,15.000000,17.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.0,2.000000,155000.000000,84.996000,2.000000,...,1.000000,1.000000,1.000000,3807.500000,17.710000,82000.000000,82.000000,906.000000,1112.000000,33.000000
50%,1.000000,3.000000,2.000000,1.000000,1.000000,2.0,2.000000,205000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4878.000000,33.180000,82000.000000,106.000000,1260.000000,1515.000000,46.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.0,2.000000,275000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6098.000000,57.250000,82000.000000,132.000000,1627.000000,1917.000000,60.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.0,1111.000000,525000.000000,121.000000,3.000000,...,2.000000,3.000000,6.000000,9980.000000,99.670000,82000.000000,210.000000,2760.000000,3079.000000,76.000000


In [32]:
#2020
clean_2020.describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,74352.000000,75710.000000,...,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000,75710.000000
mean,1.185444,9.617514,1.965632,1.389420,1.008731,1.999921,2.010910,227911.768591,90.927345,2.099023,...,1.143997,1.070493,1.723524,5044.849042,34.726299,84487.956677,111.684229,1343.913723,1579.779488,44.710474
std,0.580089,21.152340,0.182174,0.639518,0.093030,0.008902,4.030938,95955.800383,8.684864,0.299489,...,0.351089,0.304782,1.146996,1624.649684,24.153076,935.012279,33.253225,531.619183,619.721650,17.988324
min,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.263000,1.000000,...,1.000000,1.000000,1.000000,447.000000,0.110000,80200.000000,21.000000,11.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,155000.000000,85.000000,2.000000,...,1.000000,1.000000,1.000000,3870.000000,15.690000,84800.000000,89.000000,973.000000,1168.000000,32.000000
50%,1.000000,2.000000,2.000000,1.000000,1.000000,2.000000,2.000000,215000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4922.000000,28.510000,84800.000000,110.000000,1317.000000,1562.000000,43.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,285000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6135.000000,49.780000,84800.000000,133.000000,1671.000000,1976.000000,58.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.000000,1111.000000,555000.000000,121.250000,3.000000,...,2.000000,3.000000,6.000000,10370.000000,96.160000,85200.000000,205.000000,2760.000000,3200.000000,76.000000


In [33]:
clean_2020.dropna().describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000,73264.0,73264.000000,73264.000000,73264.000000,73264.000000,...,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000,73264.000000
mean,1.179788,9.441022,1.965003,1.368080,1.004914,2.0,2.011725,228588.256169,90.911293,2.096173,...,1.147098,1.071686,1.710608,5043.087601,34.759596,84500.229308,111.730304,1342.681699,1576.306535,44.767389
std,0.572064,20.906899,0.183773,0.613701,0.069926,0.0,4.097619,94895.934756,8.669274,0.295616,...,0.354206,0.307056,1.109662,1625.033912,24.106955,924.377831,33.276725,531.643896,618.868004,17.998856
min,1.000000,0.000000,1.000000,1.000000,1.000000,2.0,1.000000,5000.000000,55.263000,1.000000,...,1.000000,1.000000,1.000000,447.000000,0.110000,80200.000000,21.000000,11.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.0,2.000000,155000.000000,85.000000,2.000000,...,1.000000,1.000000,1.000000,3870.000000,15.830000,84800.000000,89.000000,973.000000,1164.000000,32.000000
50%,1.000000,2.000000,2.000000,1.000000,1.000000,2.0,2.000000,225000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4922.000000,28.590000,84800.000000,110.000000,1314.000000,1561.000000,43.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.0,2.000000,285000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6135.000000,49.780000,84800.000000,133.000000,1668.000000,1976.000000,58.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.0,1111.000000,555000.000000,121.250000,3.000000,...,2.000000,3.000000,6.000000,10370.000000,96.160000,85200.000000,205.000000,2760.000000,3200.000000,76.000000


In [34]:
#2021
clean_2021.describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,87623.000000,89221.000000,...,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000,89221.000000
mean,1.179644,11.873057,1.959001,1.361271,1.009168,1.999922,1.996559,248007.363737,90.466175,2.092646,...,1.160433,1.077538,1.737999,4951.219444,36.837649,86721.714619,110.088993,1295.886305,1533.675458,45.876789
std,0.571856,23.784649,0.198290,0.612817,0.095312,0.008857,0.058558,105551.659992,8.838774,0.290440,...,0.367009,0.312369,3.895956,1591.552359,25.585112,978.166434,35.154613,528.892663,614.894122,18.091371
min,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.253000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,82600.000000,15.000000,17.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,165000.000000,80.054500,2.000000,...,1.000000,1.000000,1.000000,3788.000000,16.560000,87100.000000,85.000000,925.000000,1125.000000,33.000000
50%,1.000000,2.000000,2.000000,1.000000,1.000000,2.000000,2.000000,235000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4835.000000,30.630000,87100.000000,108.000000,1281.000000,1529.000000,44.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,315000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6076.000000,52.790000,87100.000000,133.000000,1638.000000,1935.000000,59.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.000000,2.000000,595000.000000,120.588000,3.000000,...,2.000000,3.000000,1111.000000,9980.000000,98.050000,87100.000000,210.000000,2720.000000,2972.000000,76.000000


In [35]:
clean_2021.dropna().describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.0,86620.000000,86620.000000,86620.000000,86620.000000,...,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000
mean,1.172154,11.904976,1.958301,1.340383,1.008797,2.0,1.997114,247994.458555,90.487315,2.088767,...,1.162988,1.078365,1.721831,4947.250670,36.931080,86734.227661,110.090072,1294.000566,1529.815770,45.949642
std,0.560959,23.795342,0.199902,0.587098,0.093380,0.0,0.053646,105324.787708,8.824238,0.284936,...,0.369357,0.313959,1.159582,1590.431478,25.592783,967.366975,35.182025,528.655625,613.573232,18.106975
min,1.000000,0.000000,1.000000,1.000000,1.000000,2.0,1.000000,5000.000000,55.253000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,82600.000000,15.000000,17.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.0,2.000000,165000.000000,80.999750,2.000000,...,1.000000,1.000000,1.000000,3782.000000,16.650000,87100.000000,85.000000,924.000000,1121.000000,33.000000
50%,1.000000,2.000000,2.000000,1.000000,1.000000,2.0,2.000000,235000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4834.000000,30.810000,87100.000000,108.000000,1279.000000,1524.000000,44.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.0,2.000000,315000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6076.000000,52.990000,87100.000000,133.000000,1637.000000,1931.000000,59.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.0,2.000000,595000.000000,120.588000,3.000000,...,2.000000,3.000000,7.000000,9980.000000,98.050000,87100.000000,210.000000,2720.000000,2972.000000,76.000000


In [36]:
#2022
clean_2022.describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,98540.000000,100731.000000,...,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.000000,100731.00000
mean,1.197953,11.705225,1.943860,1.388232,1.006125,1.999950,1.995334,250835.542187,90.077289,2.100535,...,1.175567,1.084492,1.808559,4785.740467,42.301140,104394.848656,103.691088,1259.285721,1529.883184,41.14781
std,0.597264,23.699294,0.230192,0.631328,0.078024,0.007045,0.068148,112076.877910,9.350689,0.301373,...,0.380453,0.323105,1.292919,1546.913279,26.870002,7267.382867,34.219782,505.904134,572.412020,20.73840
min,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,5000.000000,55.260000,1.000000,...,1.000000,1.000000,1.000000,667.000000,3.710000,84100.000000,13.000000,7.000000,16.000000,0.00000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.000000,2.000000,165000.000000,80.000000,2.000000,...,1.000000,1.000000,1.000000,3694.000000,21.160000,105700.000000,79.000000,898.000000,1151.000000,26.00000
50%,1.000000,2.000000,2.000000,1.000000,1.000000,2.000000,2.000000,235000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4683.000000,35.190000,105700.000000,102.000000,1248.000000,1535.000000,44.00000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.000000,2.000000,325000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,5865.000000,59.820000,105700.000000,125.000000,1588.000000,1915.000000,58.00000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.000000,2.000000,645000.000000,121.037000,3.000000,...,2.000000,3.000000,7.000000,9050.000000,100.000000,118200.000000,200.000000,2572.000000,2990.000000,80.00000


In [37]:
clean_2021.dropna().describe()

,action_taken,purchaser_type,preapproval,loan_type,lien_status,reverse_mortgage,open-end_line_of_credit,loan_amount,loan_to_value_ratio,hoepa_status,...,submission_of_application,initially_payable_to_institution,aus-1,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
count,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.0,86620.000000,86620.000000,86620.000000,86620.000000,...,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000,86620.000000
mean,1.172154,11.904976,1.958301,1.340383,1.008797,2.0,1.997114,247994.458555,90.487315,2.088767,...,1.162988,1.078365,1.721831,4947.250670,36.931080,86734.227661,110.090072,1294.000566,1529.815770,45.949642
std,0.560959,23.795342,0.199902,0.587098,0.093380,0.0,0.053646,105324.787708,8.824238,0.284936,...,0.369357,0.313959,1.159582,1590.431478,25.592783,967.366975,35.182025,528.655625,613.573232,18.106975
min,1.000000,0.000000,1.000000,1.000000,1.000000,2.0,1.000000,5000.000000,55.253000,1.000000,...,1.000000,1.000000,1.000000,252.000000,0.110000,82600.000000,15.000000,17.000000,0.000000,0.000000
25%,1.000000,1.000000,2.000000,1.000000,1.000000,2.0,2.000000,165000.000000,80.999750,2.000000,...,1.000000,1.000000,1.000000,3782.000000,16.650000,87100.000000,85.000000,924.000000,1121.000000,33.000000
50%,1.000000,2.000000,2.000000,1.000000,1.000000,2.0,2.000000,235000.000000,95.000000,2.000000,...,1.000000,1.000000,1.000000,4834.000000,30.810000,87100.000000,108.000000,1279.000000,1524.000000,44.000000
75%,1.000000,6.000000,2.000000,2.000000,1.000000,2.0,2.000000,315000.000000,96.500000,2.000000,...,1.000000,1.000000,2.000000,6076.000000,52.990000,87100.000000,133.000000,1637.000000,1931.000000,59.000000
max,3.000000,72.000000,2.000000,4.000000,2.000000,2.0,2.000000,595000.000000,120.588000,3.000000,...,2.000000,3.000000,7.000000,9980.000000,98.050000,87100.000000,210.000000,2720.000000,2972.000000,76.000000
